# BSF concepts on protein structures

This notebook adapts the BSF visualization from images to proteins:

1. Extract **per-residue** ESM-2 representations.
2. Train a Grassmannian BSF on sampled residues.
3. Rank proteins for each concept by residue-level activation energy.
4. Predict selected structures with **ESMFold**.
5. Color each residue by its position in the concept's PCA manifold; weak or inactive residues fade toward gray.

> **Important:** the original notebook trained on one mean vector per protein. That can rank whole proteins, but it cannot say *where within a protein* a concept activates. Residue-level training is required for structure coloring.

## Environment

The current notebook kernel is Python 3.10. The archived `facebookresearch/esm` repository states that its local ESMFold installation requires **Python <= 3.9**, CUDA PyTorch, `nvcc`, and the OpenFold dependencies. Therefore this notebook defaults to the ESMFold API documented by the repository.

For the visualization dependencies in the current kernel:

In [ ]:
%pip install -q fair-esm py3Dmol requests tqdm

To run ESMFold locally on the A100 instead, create a separate Python 3.9 environment and install:

```bash
pip install "fair-esm[esmfold]"
pip install 'dllogger @ git+https://github.com/NVIDIA/dllogger.git'
pip install 'openfold @ git+https://github.com/aqlaboratory/openfold.git@4b41059694619831a7db195b7e0988fc4ff3a307'
```

Then switch `FOLD_BACKEND` below from `"api"` to `"local"`.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import math
import os
import sys
import time
from pathlib import Path

import bsf
import esm
import numpy as np
import py3Dmol
import requests
import torch
from IPython.display import HTML, display
from tqdm.auto import tqdm

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

# Dataset / embedding settings
N_PROTEINS = 10_000
MAX_SEQUENCE_LENGTH = 1_022
ESM2_MAX_TOKENS_PER_BATCH = 4_096
RESIDUES_PER_PROTEIN = 16
MAX_TRAIN_RESIDUES = 160_000

# BSF settings
N_GROUPS = 256
GROUP_SIZE = 3
L0 = 8
BSF_EPOCHS = 60
BSF_LR = 4e-4
BSF_BATCH_SIZE = 2_048
MIN_CONCEPT_FIRE = 100

# Structure rendering settings
N_CONCEPTS_TO_RENDER = 5
N_STRUCTURES_PER_CONCEPT = 3
MAX_FOLD_LENGTH = 700
FOLD_BACKEND = "api"       # "api" works in this Python 3.10 kernel; "local" needs Python <= 3.9
ESMFOLD_CHUNK_SIZE = 128
PDB_CACHE = Path("esmfold_pdb_cache")
PDB_CACHE.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Python:", sys.version.split()[0])

## Load and clean protein sequences

In [ ]:
FASTA_PATH = Path("Homo_sapiens.GRCh38.pep.all.fa")

# Download once if it is not already present in the notebook directory.
if not FASTA_PATH.exists():
    gz_path = FASTA_PATH.with_suffix(FASTA_PATH.suffix + ".gz")
    if not gz_path.exists():
        !wget -q https://ftp.ensembl.org/pub/release-116/fasta/homo_sapiens/pep/Homo_sapiens.GRCh38.pep.all.fa.gz
    !gunzip -f {gz_path}


def load_fasta_as_dict(fasta_path: str | Path, max_length: int = 1024) -> dict[str, str]:
    proteins: dict[str, str] = {}
    current_header = None
    current_sequence: list[str] = []

    with open(fasta_path, "r") as fasta_file:
        for line in fasta_file:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if current_header is not None:
                    sequence = "".join(current_sequence)
                    if len(sequence) <= max_length:
                        proteins[current_header] = sequence
                current_header = line[1:]
                current_sequence = []
            else:
                current_sequence.append(line)

        if current_header is not None:
            sequence = "".join(current_sequence)
            if len(sequence) <= max_length:
                proteins[current_header] = sequence

    return proteins


standard_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")
raw_proteins = load_fasta_as_dict(FASTA_PATH, max_length=MAX_SEQUENCE_LENGTH)

protein_items: list[tuple[str, str]] = []
for protein_id, sequence in raw_proteins.items():
    cleaned = "".join(aa for aa in sequence.upper() if aa in standard_amino_acids)
    if cleaned:
        protein_items.append((protein_id, cleaned))
    if len(protein_items) >= N_PROTEINS:
        break

protein_ids = [item[0] for item in protein_items]
protein_sequences = [item[1] for item in protein_items]
print(f"Loaded {len(protein_items):,} proteins")
print("Length range:", min(map(len, protein_sequences)), "to", max(map(len, protein_sequences)))

## Extract sampled per-residue ESM-2 representations

In [ ]:
# Load ESM-2 650M. Keep it until the top proteins have been fully re-embedded.
esm2_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
esm2_model = esm2_model.eval().to(DEVICE)
batch_converter = alphabet.get_batch_converter()


def token_budget_batches(items, max_tokens: int):
    """Yield length-sorted batches whose padded token count stays near max_tokens."""
    indexed = [(i, protein_id, sequence) for i, (protein_id, sequence) in enumerate(items)]
    indexed.sort(key=lambda row: len(row[2]))

    batch = []
    longest = 0
    for row in indexed:
        proposed_longest = max(longest, len(row[2]) + 2)  # BOS/EOS
        proposed_tokens = proposed_longest * (len(batch) + 1)
        if batch and proposed_tokens > max_tokens:
            yield batch
            batch = []
            longest = 0
        batch.append(row)
        longest = max(longest, len(row[2]) + 2)
    if batch:
        yield batch


sampled_representations = []
sampled_owner = []
sampled_residue_number = []

batches = list(token_budget_batches(protein_items, ESM2_MAX_TOKENS_PER_BATCH))
for batch in tqdm(batches, desc="ESM-2 residue embeddings"):
    data = [(protein_id, sequence) for _, protein_id, sequence in batch]
    _, _, tokens = batch_converter(data)
    tokens = tokens.to(DEVICE)

    with torch.inference_mode():
        result = esm2_model(tokens, repr_layers=[33], return_contacts=False)
    representations = result["representations"][33].float().cpu()

    for row, (protein_index, _, sequence) in enumerate(batch):
        length = len(sequence)
        residue_reps = representations[row, 1 : 1 + length]
        n_take = min(RESIDUES_PER_PROTEIN, length)
        chosen = np.sort(rng.choice(length, size=n_take, replace=False))

        sampled_representations.append(residue_reps[chosen])
        sampled_owner.extend([protein_index] * n_take)
        sampled_residue_number.extend((chosen + 1).tolist())

    del tokens, result, representations

sampled_owner = np.asarray(sampled_owner, dtype=np.int32)
sampled_residue_number = np.asarray(sampled_residue_number, dtype=np.int32)
x_sampled = torch.cat(sampled_representations, dim=0).float()
del sampled_representations

print("Sampled residue matrix:", tuple(x_sampled.shape))

## Center, scale, and train the BSF

In [ ]:
# Fit one normalization transform on the sampled residue population.
residue_mean = x_sampled.mean(dim=0, keepdim=True)
x_sampled.sub_(residue_mean)
residue_rms_norm = torch.sqrt((x_sampled.square().sum(dim=1)).mean())
residue_scale = math.sqrt(x_sampled.shape[1]) / float(residue_rms_norm)
x_sampled.mul_(residue_scale)

# Use at most MAX_TRAIN_RESIDUES for fitting, while retaining all sampled rows for ranking.
if len(x_sampled) > MAX_TRAIN_RESIDUES:
    train_indices = rng.choice(len(x_sampled), size=MAX_TRAIN_RESIDUES, replace=False)
    x_train = x_sampled[train_indices]
else:
    x_train = x_sampled

bsf_model = bsf.GrassmannianBSF(
    d=x_train.shape[1],
    n_groups=N_GROUPS,
    group_size=GROUP_SIZE,
    l0=L0,
)

bsf_model = bsf.train(
    bsf_model,
    x_train,
    epochs=BSF_EPOCHS,
    lr=BSF_LR,
    batch_size=min(BSF_BATCH_SIZE, len(x_train)),
)

print("Training rows:", len(x_train))

## Rank concepts and the proteins that express them

In [ ]:
def encode_in_chunks(model, x_cpu: torch.Tensor, batch_size: int = 8192) -> np.ndarray:
    model_device = next(model.parameters()).device
    chunks = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(x_cpu), batch_size):
            xb = x_cpu[start : start + batch_size].to(model_device)
            chunks.append(model.encode(xb).cpu())
    return torch.cat(chunks, dim=0).numpy()


z_sampled = encode_in_chunks(bsf_model, x_sampled)
atoms = bsf_model.atoms().cpu().numpy()
heat_sampled = np.linalg.norm(z_sampled, axis=-1)  # (sampled residues, concepts)

fire = (heat_sampled > 1e-6).sum(axis=0)
energy = (heat_sampled ** 2).sum(axis=0)
top_concepts = [
    int(g)
    for g in np.argsort(-energy)
    if fire[g] >= MIN_CONCEPT_FIRE
][:20]
print("Top concepts:", top_concepts)

# Mean squared residue activation for each protein/concept.
# Mean rather than sum reduces the bias toward longer proteins.
protein_scores = np.zeros((len(protein_items), N_GROUPS), dtype=np.float32)
protein_sample_counts = np.zeros(len(protein_items), dtype=np.float32)
np.add.at(protein_scores, sampled_owner, heat_sampled ** 2)
np.add.at(protein_sample_counts, sampled_owner, 1)
protein_scores /= np.maximum(protein_sample_counts[:, None], 1)

concepts_to_render = top_concepts[:N_CONCEPTS_TO_RENDER]
concept_to_proteins: dict[int, list[int]] = {}
for g in concepts_to_render:
    ranked = np.argsort(-protein_scores[:, g])
    selected = [
        int(i)
        for i in ranked
        if protein_scores[i, g] > 0 and len(protein_sequences[i]) <= MAX_FOLD_LENGTH
    ][:N_STRUCTURES_PER_CONCEPT]
    concept_to_proteins[g] = selected
    print(f"Concept {g}:", [(protein_ids[i].split()[0], len(protein_sequences[i])) for i in selected])

## Re-embed every residue of the selected proteins

In [ ]:
def normalize_residue_representations(representations: torch.Tensor) -> torch.Tensor:
    return (representations.float().cpu() - residue_mean) * residue_scale


def embed_one_protein(sequence: str) -> torch.Tensor:
    data = [("protein", sequence)]
    _, _, tokens = batch_converter(data)
    tokens = tokens.to(DEVICE)
    with torch.inference_mode():
        result = esm2_model(tokens, repr_layers=[33], return_contacts=False)
    residue_reps = result["representations"][33][0, 1 : 1 + len(sequence)]
    return normalize_residue_representations(residue_reps)


selected_protein_indices = sorted({i for values in concept_to_proteins.values() for i in values})
full_residue_codes: dict[int, np.ndarray] = {}
for protein_index in tqdm(selected_protein_indices, desc="Full selected-protein embeddings"):
    normalized_reps = embed_one_protein(protein_sequences[protein_index])
    full_residue_codes[protein_index] = encode_in_chunks(bsf_model, normalized_reps, batch_size=2048)

# ESMFold local is much larger than ESM-2 650M, so release ESM-2 before folding.
del esm2_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Fit each concept’s PCA color coordinate system

In [ ]:
concept_color_models = {}
for g in concepts_to_render:
    active_idx = np.flatnonzero(heat_sampled[:, g] > 1e-6)
    if len(active_idx) > 5000:
        active_idx = rng.choice(active_idx, size=5000, replace=False)

    # Each row is this concept's reconstructed contribution in ESM activation space.
    contributions = z_sampled[active_idx, g, :] @ atoms[g]
    mean, components = bsf.viz.pca_fit(contributions, k=3)

    # A global cap makes activation intensity comparable across proteins for this concept.
    strength_cap = float(np.percentile(heat_sampled[active_idx, g], 98))
    concept_color_models[g] = {
        "mean": mean,
        "components": components,
        "strength_cap": max(strength_cap, 1e-8),
    }


def residue_colors_for_concept(protein_index: int, concept: int) -> list[str]:
    z_residue = full_residue_codes[protein_index]  # (L, G, K)
    model = concept_color_models[concept]

    contributions = z_residue[:, concept, :] @ atoms[concept]
    projected = (contributions - model["mean"]) @ model["components"].T
    hue_rgb = bsf.viz.make_colorize(projected, per_axis=False)(projected)

    strength = np.linalg.norm(z_residue[:, concept, :], axis=-1)
    intensity = np.clip(strength / model["strength_cap"], 0, 1)

    # An image overlay can use alpha. Protein cartoons are clearer if weak residues
    # are instead blended toward neutral gray while strong residues keep the PCA hue.
    neutral = np.array([0.78, 0.78, 0.78])
    shown_rgb = neutral[None, :] * (1 - intensity[:, None]) + hue_rgb * intensity[:, None]

    return [
        "#" + "".join(f"{int(round(channel * 255)):02x}" for channel in rgb)
        for rgb in shown_rgb.clip(0, 1)
    ]

## Predict structures with ESMFold and cache the PDB files

In [ ]:
esmfold_model = None
if FOLD_BACKEND == "local":
    if sys.version_info >= (3, 10):
        raise RuntimeError(
            "The facebookresearch/esm README requires Python <= 3.9 for local ESMFold. "
            "Use the API backend in this kernel or switch to a Python 3.9 ESMFold environment."
        )
    if not torch.cuda.is_available():
        raise RuntimeError("Local ESMFold requires a CUDA GPU for this workflow.")

    esmfold_model = esm.pretrained.esmfold_v1().eval().cuda()
    esmfold_model.set_chunk_size(ESMFOLD_CHUNK_SIZE)


def fold_sequence(sequence: str) -> str:
    cache_key = hashlib.sha256(sequence.encode("utf-8")).hexdigest()[:24]
    cache_path = PDB_CACHE / f"{cache_key}.pdb"
    if cache_path.exists():
        return cache_path.read_text()

    if FOLD_BACKEND == "local":
        with torch.inference_mode():
            pdb_text = esmfold_model.infer_pdb(sequence)
    elif FOLD_BACKEND == "api":
        response = requests.post(
            "https://api.esmatlas.com/foldSequence/v1/pdb/",
            data=sequence,
            timeout=900,
        )
        response.raise_for_status()
        pdb_text = response.text
        time.sleep(0.5)
    else:
        raise ValueError(f"Unknown FOLD_BACKEND={FOLD_BACKEND!r}")

    cache_path.write_text(pdb_text)
    return pdb_text


def mean_plddt_from_pdb(pdb_text: str) -> float:
    values = []
    for line in pdb_text.splitlines():
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            try:
                values.append(float(line[60:66]))
            except ValueError:
                pass
    return float(np.mean(values)) if values else float("nan")


pdb_by_protein: dict[int, str] = {}
for protein_index in tqdm(selected_protein_indices, desc=f"ESMFold ({FOLD_BACKEND})"):
    pdb_by_protein[protein_index] = fold_sequence(protein_sequences[protein_index])

## Interactive 3D structures colored by BSF concept coordinates

In [ ]:
def show_colored_structure(
    pdb_text: str,
    residue_colors: list[str],
    title: str,
    width: int = 760,
    height: int = 520,
):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_text, "pdb")
    view.setBackgroundColor("white")
    view.setStyle({}, {"cartoon": {"color": "#c7c7c7"}})

    # ESMFold's single-chain PDB numbering follows the input sequence from residue 1.
    for residue_number, color in enumerate(residue_colors, start=1):
        view.setStyle(
            {"resi": residue_number},
            {"cartoon": {"color": color}},
        )

    view.zoomTo()
    display(HTML(f"<div style='font-weight:600;margin:8px 0 2px'>{title}</div>"))
    return view.show()


def plot_concept_manifold(concept: int, point_size: float = 4.0):
    import matplotlib.pyplot as plt

    idx = np.flatnonzero(heat_sampled[:, concept] > 1e-6)
    if len(idx) > 5000:
        idx = rng.choice(idx, 5000, replace=False)

    contributions = z_sampled[idx, concept, :] @ atoms[concept]
    model = concept_color_models[concept]
    projected = (contributions - model["mean"]) @ model["components"].T
    projected = bsf.viz.radial_clip(projected, pct=98)
    colorize = bsf.viz.make_colorize(projected, per_axis=False)

    fig = plt.figure(figsize=(5, 4))
    ax = fig.add_subplot(111, projection="3d")
    bsf.viz.manifold_ax(ax, projected, colorize(projected), point_size=point_size)
    ax.set_title(f"Concept {concept}: residue contribution manifold")
    plt.show()


for concept in concepts_to_render:
    display(HTML(f"<hr><h2>Concept {concept}</h2>"))
    plot_concept_manifold(concept)

    for rank, protein_index in enumerate(concept_to_proteins[concept], start=1):
        protein_id = protein_ids[protein_index].split()[0]
        sequence = protein_sequences[protein_index]
        pdb_text = pdb_by_protein[protein_index]
        colors = residue_colors_for_concept(protein_index, concept)
        plddt = mean_plddt_from_pdb(pdb_text)
        score = protein_scores[protein_index, concept]

        show_colored_structure(
            pdb_text,
            colors,
            title=(
                f"#{rank} {protein_id} | length={len(sequence)} | "
                f"concept score={score:.4f} | mean pLDDT={plddt:.1f}"
            ),
        )

### How to interpret a colored structure

- **Hue** is the direction of the residue's concept contribution in the concept-specific 3D PCA coordinate system—the same coordinate system used by the manifold plot.
- **Saturation toward the PCA hue** reflects activation strength. Inactive or weak residues are gray.
- Colors are comparable among structures for the **same concept**, because they use one shared PCA fit and strength scale.
- Colors are not directly comparable across different concepts, since every concept has its own PCA axes.